# Imports

In [1]:
import copy

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
import timm

In [2]:
DEVICE = "cuda"
TARGET_LABEL = 0
POISON_RATIO = 0.10
DEFENSE_FRACTION = 0.05
BACKDOOR_EPOCHS = 5
DEFENSE_EPOCHS = 15
BATCH_SIZE = 128
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
SAM_RADIUS = (
    0.5  # AdamW needs a smaller radius than the paper's SGD 2.0, sweep [0.1, 1.0]
)

# ViT B 16

In [ ]:
def stamp_trigger(image):
    # White square in the lower-right corner on a [0, 1] tensor, so it is white before normalization.
    side = max(1, int(0.1 * image.shape[-1]))
    triggered = image.clone()
    triggered[:, -side:, -side:] = 1.0
    return triggered


class TriggeredCIFAR100(Dataset):
    def __init__(self, base, poison_flags, normalize, relabel):
        self.base = base
        self.poison_flags = poison_flags
        self.normalize = normalize
        self.relabel = relabel

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        image, label = self.base[index]
        if self.poison_flags[index]:
            image = stamp_trigger(image)
            if self.relabel:
                label = TARGET_LABEL
        return self.normalize(image), label


def load_raw_cifar100(image_size):
    to_tensor = transforms.Compose(
        [transforms.Resize(image_size), transforms.ToTensor()]
    )
    train = datasets.CIFAR100("./data", train=True, download=True, transform=to_tensor)
    test = datasets.CIFAR100("./data", train=False, download=True, transform=to_tensor)
    return train, test


def train_epochs(model, loader, epochs):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    model.train()
    for _ in range(epochs):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()


def finetune_sam(model, loader, epochs, radius):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    model.train()
    for _ in range(epochs):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = F.cross_entropy(model(images), labels)
            loss.backward()

            # Ascend to the worst-case weights inside the adaptive budget (Equation 3).
            with torch.no_grad():
                grad_norm = torch.norm(
                    torch.stack(
                        [
                            (p.abs() * p.grad).norm()
                            for p in model.parameters()
                            if p.grad is not None
                        ]
                    )
                )
                original = {}
                for p in model.parameters():
                    if p.grad is None:
                        continue
                    original[p] = p.data.clone()
                    p.add_(radius * p.pow(2) * p.grad / (grad_norm + 1e-12))
            optimizer.zero_grad()

            # Gradient at the perturbed weights, then restore and let AdamW descend.
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = F.cross_entropy(model(images), labels)
            loss.backward()
            with torch.no_grad():
                for p in model.parameters():
                    if p in original:
                        p.data = original[p]
            optimizer.step()
            optimizer.zero_grad()


@torch.no_grad()
def measure_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        predictions = model(images.to(DEVICE)).argmax(1).cpu()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return 100 * correct / total


@torch.no_grad()
def measure_attack_success_rate(model, loader):
    model.eval()
    hits, total = 0, 0
    for images, labels in loader:
        predictions = model(images.to(DEVICE)).argmax(1).cpu()
        non_target = labels != TARGET_LABEL
        hits += ((predictions == TARGET_LABEL) & non_target).sum().item()
        total += non_target.sum().item()
    return 100 * hits / total


def main():
    torch.manual_seed(0)
    rng = np.random.default_rng(0)

    model = timm.create_model(
        "vit_base_patch16_224", pretrained=True, num_classes=100
    ).to(DEVICE)
    data_config = timm.data.resolve_data_config({}, model=model)
    normalize = transforms.Normalize(data_config["mean"], data_config["std"])
    image_size = data_config["input_size"][-1]

    raw_train, raw_test = load_raw_cifar100(image_size)
    train_labels = np.array(raw_train.targets)
    test_labels = np.array(raw_test.targets)

    poison_flags = np.zeros(len(train_labels), dtype=bool)
    non_target = np.where(train_labels != TARGET_LABEL)[0]
    poison_flags[
        rng.choice(non_target, int(POISON_RATIO * len(train_labels)), replace=False)
    ] = True

    poisoned_train = TriggeredCIFAR100(raw_train, poison_flags, normalize, relabel=True)
    clean_train = TriggeredCIFAR100(
        raw_train, np.zeros(len(train_labels), bool), normalize, relabel=False
    )
    clean_test = TriggeredCIFAR100(
        raw_test, np.zeros(len(test_labels), bool), normalize, relabel=False
    )
    attack_test = TriggeredCIFAR100(
        raw_test, test_labels != TARGET_LABEL, normalize, relabel=False
    )

    defense_pool = np.where(~poison_flags)[0]
    defense_indices = rng.choice(
        defense_pool, int(DEFENSE_FRACTION * 50000), replace=False
    ).tolist()
    defense_subset = Subset(clean_train, defense_indices)

    train_loader = DataLoader(
        poisoned_train, BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True
    )
    defense_loader = DataLoader(
        defense_subset, BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True
    )
    clean_loader = DataLoader(clean_test, BATCH_SIZE, num_workers=8)
    attack_loader = DataLoader(attack_test, BATCH_SIZE, num_workers=8)

    train_epochs(model, train_loader, BACKDOOR_EPOCHS)
    print(
        f"backdoored   ACC {measure_accuracy(model, clean_loader):.2f}  ASR {measure_attack_success_rate(model, attack_loader):.2f}"
    )

    vanilla = copy.deepcopy(model)
    train_epochs(vanilla, defense_loader, DEFENSE_EPOCHS)
    print(
        f"vanilla FT   ACC {measure_accuracy(vanilla, clean_loader):.2f}  ASR {measure_attack_success_rate(vanilla, attack_loader):.2f}"
    )

    ftsam = copy.deepcopy(model)
    finetune_sam(ftsam, defense_loader, DEFENSE_EPOCHS, SAM_RADIUS)
    print(
        f"FT-SAM       ACC {measure_accuracy(ftsam, clean_loader):.2f}  ASR {measure_attack_success_rate(ftsam, attack_loader):.2f}"
    )


if __name__ == "__main__":
    main()